# LoveDA UDA Brainstorm

Focus: `SegFormer-MiT-B2` backbone with a `Teacher-Student Consistency` UDA setup.


## Goal

Build an implementation-ready training plan before writing the actual UDA pipeline.

Target assumptions:
- source domain: labeled LoveDA split
- target domain: unlabeled domain for UDA
- segmentation backbone: SegFormer-MiT-B2
- adaptation method: EMA teacher + student consistency on target images


In [3]:
from pathlib import Path
import importlib
import torch

ROOT = Path('..')
DATA_ROOT = ROOT / 'data' / 'LoveDA'
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('dataset exists:', DATA_ROOT.exists())


torch: 2.12.0+cu130
cuda available: True
dataset exists: True


## Proposed UDA setup

### Model
- student: `SegFormer-MiT-B2`
- teacher: EMA copy of the student
- decoder head: standard segmentation head with `num_classes=8` and `ignore_index=0` for supervised source loss

### Source branch
- input: source image + source mask
- loss: supervised CE or CE+Dice

### Target branch
- weakly augmented target image -> teacher prediction
- strongly augmented target image -> student prediction
- consistency loss only on confident teacher pixels


## Candidate losses

Total loss:
```text
L = L_source_supervised + lambda_u * L_target_consistency
```

Recommended starting point:
- `L_source_supervised = CrossEntropyLoss(ignore_index=0)`
- optional later: `CE + Dice`
- `L_target_consistency = masked CE or KL between teacher/student soft predictions`
- confidence mask from teacher max probability, e.g. `p_max >= 0.95` initially


## Important implementation choices

1. Pair each source batch with a target batch.
2. Keep teacher in eval mode, update via EMA only.
3. Use weak/strong augmentation split only on target branch.
4. Exclude invalid / no-data regions where possible.
5. Report per-domain validation metrics.


In [4]:
# Minimal environment probe for a future SegFormer implementation
for module_name in ['torch', 'torchvision', 'transformers', 'timm']:
    spec = importlib.util.find_spec(module_name)
    print(module_name, 'installed' if spec is not None else 'missing')


torch installed
torchvision installed
transformers installed
timm installed


## Immediate coding roadmap

1. Define class ids/constants in `src/utils/constants.py`
2. Implement LoveDA dataset loader in `src/datasets/loveda.py`
3. Add SegFormer-MiT-B2 model wrapper in `src/models/segformer.py`
4. Add source/target augmentation policy
5. Build a notebook or script-level dry run for forward pass and loss computation
6. Only then write full training loop


## Dry run with current repo modules

This section checks that the current dataset and SegFormer wrapper can support the planned UDA pipeline.


In [ ]:
from src.datasets import LoveDADataset
from src.models import SegFormerMiTB2, create_teacher_from_student, update_ema_teacher
from src.utils import LOVE_DA_ROOT

dataset = LoveDADataset(
    dataset_root=LOVE_DA_ROOT,
    splits=('Train',),
    domains=('Urban',),
    require_masks=True,
    ignore_black_padding=True,
)
sample = dataset[0]
sample['meta']


In [ ]:
model = SegFormerMiTB2(num_classes=8, ignore_index=0, use_pretrained=False)
teacher = create_teacher_from_student(model)

x = sample['image'].unsqueeze(0)
with torch.no_grad():
    logits = model.forward_logits(x)
logits.shape


## UDA training-step sketch

Expected training structure:

1. source batch -> supervised loss
2. target weak batch -> teacher pseudo-labels
3. target strong batch -> student prediction
4. confidence mask -> consistency loss
5. student optimizer step
6. EMA teacher update
